# Where does HuBERT encode articulation? Layer-wise probing against EMA

**Model:** HuBERT Base (`facebook/hubert-base-ls960`) · **Data:** MOCHA-TIMIT, electromagnetic articulography (EMA) at 500 Hz + 16 kHz audio · **Runs on:** Colab free tier (T4 GPU)

**Question.** Self-supervised speech models are trained only on audio, never on how speech is produced. Do their internal layers nonetheless encode articulator movements (jaw, lips, tongue, velum), and in which layers is that information strongest?

**Method.** For every layer, train a *linear* probe (ridge regression) to predict 14 articulator trajectories from that layer's frame-level features, and score it by the correlation between predicted and measured trajectories on held-out sentences. A linear probe can only read out information that is already explicit in the features, so high correlation means the layer represents articulation in an accessible form.

**Controls.**
- **MFCC baseline:** classic acoustic features (13 MFCCs + deltas, ±5 frames of context). If HuBERT can't beat this, it hasn't learned anything about articulation beyond the raw acoustics.
- **Random-init HuBERT:** the same architecture with untrained weights. This separates what HuBERT *learned* from what its architecture and input give for free.

This is a small-scale replication in the spirit of Cho et al. (ICASSP 2023), *Evidence of Vocal Tract Articulation in Self-Supervised Learning of Speech*, and the first step towards analysing individual sparse-autoencoder features on the best layer.

## Predictions (written before running)

1. **(P1)** The best pretrained HuBERT layer predicts articulator trajectories better than the MFCC baseline.
2. **(P2)** Pretrained HuBERT beats the random-init model at the same layer, at most layers. That is, the articulatory information is learned, not built in.
3. **(P3)** Articulatory information peaks in the middle layers, not the final layer.

If P1 or P2 fails, HuBERT's apparent articulatory knowledge would just be the acoustics, or an architectural artifact.

**Runtime on Colab free tier (T4 GPU):** download under a minute, feature extraction a few minutes, probes a few minutes. Large feature files live on Colab's local disk; only results are saved to Google Drive.

## Setup

In [ ]:
!pip -q install transformers soundfile librosa

In [ ]:
import os, glob, json, tarfile, urllib.request
import numpy as np
import pandas as pd
import torch
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
from scipy import signal, stats
from tqdm.auto import tqdm
from IPython.display import display

torch.set_grad_enabled(False)
SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

MODEL_NAME = "facebook/hubert-base-ls960"
SPEAKER = "fsew0"                       # MOCHA-TIMIT speakers: "fsew0" (female) or "msak0" (male)
MOCHA_URL = "https://data.cstr.ed.ac.uk/mocha"
DATA_DIR = "/content/mocha"
WORK_DIR = "/content/work"              # large feature files (Colab local disk)
RESULTS_DIR = "/content/drive/MyDrive/hubert_probing"   # small results (Google Drive)

# Articulators to predict (x and y for each): jaw (lower incisor), upper lip, lower lip,
# tongue tip, tongue blade, tongue dorsum, velum. Reference coils are left out.
ARTICULATORS = ["li", "ul", "ll", "tt", "tb", "td", "v"]
ARTICULATOR_NAMES = {"li": "jaw", "ul": "upper lip", "ll": "lower lip", "tt": "tongue tip",
                     "tb": "tongue blade", "td": "tongue dorsum", "v": "velum"}

LOWPASS_HZ = 20                 # articulator movement is slow; filter EMA noise above this
REMOVE_UTTERANCE_MEAN = True    # removes slow drift / head-position offsets between sentences
MFCC_CONTEXT = 5                # +-5 frames (+-100 ms) of context for the MFCC baseline
ALPHAS = [10.0 ** k for k in range(-1, 7)]
SPLIT = (0.8, 0.1, 0.1)         # train / validation / test, split by sentence

FEATURE_COLOR, RANDOM_COLOR, MFCC_COLOR = "#2a78d6", "#8a8a85", "#eb6834"

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Google Drive not mounted, so results stay on local disk:", e)
    RESULTS_DIR = os.path.join(WORK_DIR, "results")
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("results ->", RESULTS_DIR)

## Download MOCHA-TIMIT

Free for non-commercial use (University of Edinburgh, CSTR). If the download fails, open the `MOCHA_URL` page in a browser, check the exact tarball name for your speaker, and set `TARBALL` below.

In [ ]:
TARBALL = f"{SPEAKER}_v1.1.tar.gz"
os.makedirs(DATA_DIR, exist_ok=True)

if not glob.glob(f"{DATA_DIR}/**/*.ema", recursive=True):
    path = os.path.join(DATA_DIR, TARBALL)
    url = f"{MOCHA_URL}/{TARBALL}"
    print("Downloading", url)
    try:
        urllib.request.urlretrieve(url, path)
    except Exception as e:
        raise RuntimeError(f"Download failed ({e}). Open {MOCHA_URL}/ in a browser, find the tarball "
                           f"for {SPEAKER}, and set TARBALL to its exact name.")
    with tarfile.open(path) as tf:
        try:
            tf.extractall(DATA_DIR, filter="data")
        except TypeError:                      # Python < 3.12
            tf.extractall(DATA_DIR)
print("data in", DATA_DIR)

## Readers

- **EMA** files are Edinburgh Speech Tools (EST) binary tracks: an ASCII header ending in `EST_Header_End`, then for every frame a time value, a break flag and one float per channel. The byte order is read from the header and double-checked by requiring the time column to increase.
- **Audio** files have 1024-byte NIST headers. `soundfile` reads these directly, and there's a manual fallback.

In [ ]:
def read_est_track(path):
    """Returns (times [n], values [n, channels], channel names) for an EST binary track."""
    with open(path, "rb") as f:
        raw = f.read()
    end = raw.index(b"EST_Header_End")
    end = raw.index(b"\n", end) + 1
    lines = raw[:end].decode("latin-1").splitlines()
    assert lines[0].split()[:2] == ["EST_File", "Track"], f"not an EST track: {path}"
    meta, names = {}, {}
    for line in lines[1:]:
        parts = line.split()
        if len(parts) < 2:
            continue
        if parts[0].startswith("Channel_"):
            names[int(parts[0].split("_")[1])] = parts[1]
        else:
            meta[parts[0]] = parts[1]
    assert meta.get("DataType", "ascii") == "binary", f"only binary EST tracks are supported: {path}"
    n_frames, n_ch = int(meta["NumFrames"]), int(meta["NumChannels"])
    body = raw[end:]

    def decode(dtype):
        return np.frombuffer(body, dtype=dtype, count=n_frames * (n_ch + 2)).reshape(n_frames, n_ch + 2)

    def plausible(d):
        t = d[:, 0].astype(np.float64)
        return np.all(np.isfinite(t)) and np.all(np.diff(t) > 0) and t[-1] < 1e4

    guess = ">f4" if meta.get("ByteOrder", "10") == "10" else "<f4"
    data = decode(guess)
    if not plausible(data):
        data = decode("<f4" if guess == ">f4" else ">f4")
    assert plausible(data), f"could not decode {path}"
    channel_names = [names.get(i, f"ch{i}") for i in range(n_ch)]
    return data[:, 0].astype(np.float64), data[:, 2:].astype(np.float32), channel_names

# Documented MOCHA channel order (used only if a file has no channel names):
# coils 1-5 x, coils 1-5 y, coils 6-10 x, coils 6-10 y.
DOC_COILS = ["ui", "li", "ul", "ll", "tt", "tb", "td", "v", "c9", "c10"]
def documented_names(n_ch):
    if n_ch != 20:
        return None
    c = DOC_COILS
    return ([f"{k}_x" for k in c[:5]] + [f"{k}_y" for k in c[:5]]
            + [f"{k}_x" for k in c[5:]] + [f"{k}_y" for k in c[5:]])

def select_channels(names):
    """Indices and labels of the x/y channels for each articulator in ARTICULATORS."""
    def match(ns):
        idx, labels = [], []
        for a in ARTICULATORS:
            for axis in ["x", "y"]:
                hits = [i for i, n in enumerate(ns) if n.lower().replace("_", "") == f"{a}{axis}"]
                if len(hits) != 1:
                    return None
                idx.append(hits[0])
                labels.append(f"{a}_{axis}")
        return idx, labels
    found = match(names)
    if found is None and documented_names(len(names)) is not None:
        print("Channel names not recognised; using the documented MOCHA channel order instead.")
        found = match(documented_names(len(names)))
    assert found is not None, (f"Could not find all of {ARTICULATORS} among channels {names}. "
                               "Edit ARTICULATORS to match these names.")
    return found

def read_nist(path):
    with open(path, "rb") as f:
        raw = f.read()
    head_lines = raw[:1024].decode("latin-1").splitlines()
    hsize = int(head_lines[1].strip())
    fields = {}
    for line in head_lines[2:]:
        parts = line.split()
        if len(parts) >= 3:
            fields[parts[0]] = parts[2]
        if line.startswith("end_head"):
            break
    dtype = ">i2" if fields.get("sample_byte_format", "01") == "10" else "<i2"
    wav = np.frombuffer(raw[hsize:], dtype=dtype).astype(np.float32) / 32768.0
    return wav, int(fields.get("sample_rate", 16000))

def read_audio(path):
    try:
        wav, sr = sf.read(path, dtype="float32")
    except Exception:
        wav, sr = read_nist(path)
    if wav.ndim > 1:
        wav = wav[:, 0]
    if sr != 16000:
        wav, sr = librosa.resample(wav, orig_sr=sr, target_sr=16000), 16000
    return wav, sr

utts = []
for ema_path in sorted(glob.glob(f"{DATA_DIR}/**/{SPEAKER}*.ema", recursive=True)):
    wav_path = ema_path[:-4] + ".wav"
    if os.path.exists(wav_path):
        utts.append(dict(id=os.path.basename(ema_path)[:-4], ema=ema_path, wav=wav_path))
print(f"{len(utts)} utterances with both audio and EMA")
assert utts, "No matching .ema/.wav pairs found."

## Inspect one utterance (sanity check)

In [ ]:
t_ema, x_ema, ch_names = read_est_track(utts[0]["ema"])
wav, sr = read_audio(utts[0]["wav"])
CH_IDX, CH_LABELS = select_channels(ch_names)
print("EMA channels in file:", ch_names)
print("Using:", CH_LABELS)
print(f"EMA rate ~{1 / np.median(np.diff(t_ema)):.0f} Hz, {t_ema[-1]:.2f} s; audio {sr} Hz, {len(wav) / sr:.2f} s")

fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
axes[0].plot(np.arange(len(wav)) / sr, wav, color=RANDOM_COLOR, lw=0.5)
axes[0].set_ylabel("audio")
j = CH_LABELS.index("tt_y")
axes[1].plot(t_ema, x_ema[:, CH_IDX[j]], color=FEATURE_COLOR)
axes[1].set_ylabel("tongue tip (y)")
axes[1].set_xlabel("time (s)")
axes[0].set_title(utts[0]["id"])
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## Models

`random` has exactly the same architecture and input processing as `pretrained`, but untrained weights.

In [ ]:
from transformers import AutoConfig, AutoFeatureExtractor, AutoModel, Wav2Vec2FeatureExtractor

try:
    feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
except Exception as e:
    print("No feature-extractor config found; using the raw waveform without normalisation:", e)
    feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, do_normalize=False)

config = AutoConfig.from_pretrained(MODEL_NAME)
torch.manual_seed(SEED)
models = {
    "pretrained": AutoModel.from_pretrained(MODEL_NAME).to(device).eval(),
    "random": AutoModel.from_config(config).to(device).eval(),
}

In [ ]:
cfg = models["pretrained"].config
N_LAYERS = cfg.num_hidden_layers + 1     # hidden_states[0] is the input to the first transformer layer
HIDDEN = cfg.hidden_size

# Frame timing from the convolutional front end (HuBERT Base: 20 ms hop, 25 ms receptive field).
HOP, RF = 1, 1
for k, s in zip(cfg.conv_kernel, cfg.conv_stride):
    RF += (k - 1) * HOP
    HOP *= s
def frame_centres(n_frames, sr=16000):
    return (np.arange(n_frames) * HOP + (RF - 1) / 2) / sr
print(f"{N_LAYERS} layers x {HIDDEN} dims; hop {HOP / 16:.0f} ms, receptive field {RF / 16:.0f} ms")

## Targets: EMA aligned to the model's frames

For each sentence: low-pass filter the EMA at 20 Hz, sample it at the centre of every model frame, optionally subtract the sentence mean, and keep only frames that fall inside the EMA recording.

In [ ]:
def n_model_frames(n_samples):
    return int(models["pretrained"]._get_feat_extract_output_lengths(torch.tensor(n_samples)))

def ema_targets(times, t_ema, x_ema):
    x = x_ema[:, CH_IDX].astype(np.float64)
    fs = 1.0 / np.median(np.diff(t_ema))
    b, a = signal.butter(5, LOWPASS_HZ / (fs / 2))
    x = signal.filtfilt(b, a, x, axis=0)
    y = np.stack([np.interp(times, t_ema, x[:, j]) for j in range(x.shape[1])], axis=1)
    if REMOVE_UTTERANCE_MEAN:
        y = y - y.mean(0)
    return y

rng = np.random.default_rng(SEED)
order = rng.permutation(len(utts))
n_tr, n_va = int(SPLIT[0] * len(utts)), int(SPLIT[1] * len(utts))
split_of = {}
for rank, i in enumerate(order):
    split_of[utts[i]["id"]] = 0 if rank < n_tr else (1 if rank < n_tr + n_va else 2)

plan, Y_parts, skipped = [], [], []
for u in tqdm(utts, desc="aligning EMA"):
    t_ema, x_ema, _ = read_est_track(u["ema"])
    if not np.all(np.isfinite(x_ema[:, CH_IDX])):
        skipped.append(u["id"])
        continue
    wav, sr = read_audio(u["wav"])
    times = frame_centres(n_model_frames(len(wav)))
    keep = np.where((times >= t_ema[0]) & (times <= t_ema[-1]))[0]
    if len(keep) < 10:
        skipped.append(u["id"])
        continue
    Y_parts.append(ema_targets(times[keep], t_ema, x_ema))
    plan.append(dict(id=u["id"], wav=u["wav"], f0=int(keep[0]), f1=int(keep[-1]) + 1,
                     n_total=len(times), split=split_of[u["id"]]))

offsets = np.cumsum([0] + [p["f1"] - p["f0"] for p in plan])
for p, o in zip(plan, offsets[:-1]):
    p["offset"] = int(o)
TOTAL = int(offsets[-1])
utt_of_frame = np.concatenate([np.full(p["f1"] - p["f0"], k) for k, p in enumerate(plan)])
split_of_frame = np.array([plan[k]["split"] for k in utt_of_frame])

Y = np.concatenate(Y_parts).astype(np.float32)
y_mu, y_sd = Y[split_of_frame == 0].mean(0), Y[split_of_frame == 0].std(0) + 1e-8
Y = (Y - y_mu) / y_sd                          # z-score with training-set statistics

print(f"{len(plan)} sentences, {TOTAL} frames ({TOTAL / 50 / 60:.1f} min of speech); skipped {len(skipped)}")
print("sentences per split (train/val/test):", np.bincount([p["split"] for p in plan], minlength=3))

## Extract layer-wise features

Each layer is written to its own float16 file on Colab's local disk, roughly 100 MB per layer per model for one speaker. A `done` marker means that, after a disconnect, finished models are skipped and only unfinished work is redone.

In [ ]:
def layer_path(kind, layer):
    return os.path.join(WORK_DIR, f"{SPEAKER}_{kind}_layer{layer:02d}.npy")

for kind, model in models.items():
    marker = os.path.join(WORK_DIR, f"{SPEAKER}_{kind}.done")
    if os.path.exists(marker):
        print(kind, "features already extracted")
        continue
    mms = [np.lib.format.open_memmap(layer_path(kind, l), mode="w+", dtype=np.float16, shape=(TOTAL, HIDDEN))
           for l in range(N_LAYERS)]
    for p in tqdm(plan, desc=f"{kind} features"):
        wav, sr = read_audio(p["wav"])
        inputs = feature_extractor(wav, sampling_rate=sr, return_tensors="pt").input_values.to(device)
        hidden = model(inputs, output_hidden_states=True).hidden_states
        assert len(hidden) == N_LAYERS and hidden[0].shape[1] == p["n_total"], "frame count mismatch"
        n = p["f1"] - p["f0"]
        for l, h in enumerate(hidden):
            mms[l][p["offset"]:p["offset"] + n] = h[0, p["f0"]:p["f1"]].float().cpu().numpy().astype(np.float16)
    for mm in mms:
        mm.flush()
    del mms
    open(marker, "w").close()
print("feature files:", len(glob.glob(os.path.join(WORK_DIR, "*.npy"))))

## MFCC baseline features

In [ ]:
def mfcc_features(wav, sr, times):
    m = librosa.feature.mfcc(y=wav, sr=sr, n_mfcc=13, n_fft=400, hop_length=160)
    feats = np.vstack([m, librosa.feature.delta(m), librosa.feature.delta(m, order=2)]).T   # [frames, 39]
    ftimes = librosa.frames_to_time(np.arange(len(feats)), sr=sr, hop_length=160)
    f = np.stack([np.interp(times, ftimes, feats[:, j]) for j in range(feats.shape[1])], axis=1)
    pad = np.pad(f, ((MFCC_CONTEXT, MFCC_CONTEXT), (0, 0)), mode="edge")
    return np.hstack([pad[i:i + len(f)] for i in range(2 * MFCC_CONTEXT + 1)]).astype(np.float32)

X_mfcc = np.concatenate([
    mfcc_features(*read_audio(p["wav"]), frame_centres(p["n_total"])[p["f0"]:p["f1"]])
    for p in tqdm(plan, desc="MFCC")])
assert len(X_mfcc) == TOTAL
print("MFCC features:", X_mfcc.shape)

## Linear probes

Ridge regression from features to all 14 articulator channels at once. Features are standardised with training-set statistics, the ridge penalty is chosen on the validation sentences, and the score is the Pearson correlation between predicted and measured trajectories on the **test** sentences.

Two scores are reported:
- **r per channel**, over all test frames together;
- **r per sentence** (the mean over channels for each test sentence), which is used for paired significance tests.

In [ ]:
Y_tr, Y_va, Y_te = (Y[split_of_frame == s] for s in range(3))
test_utts = [k for k, p in enumerate(plan) if p["split"] == 2]

def corr_cols(a, b):
    a = a - a.mean(0)
    b = b - b.mean(0)
    denom = np.sqrt((a ** 2).sum(0) * (b ** 2).sum(0))
    return np.where(denom > 0, (a * b).sum(0) / np.where(denom > 0, denom, 1), np.nan)

def probe(X):
    X = np.asarray(X, dtype=np.float32)
    X_tr, X_va, X_te = (X[split_of_frame == s] for s in range(3))
    mu, sd = X_tr.mean(0), X_tr.std(0) + 1e-6
    Z = ((X_tr - mu) / sd).astype(np.float64)
    evals, V = np.linalg.eigh(Z.T @ Z)
    VC = V.T @ (Z.T @ (Y_tr - Y_tr.mean(0)))
    def predict(X_, alpha):
        W = V @ (VC / (evals + alpha)[:, None])
        return ((X_ - mu) / sd) @ W + Y_tr.mean(0)
    best = max(ALPHAS, key=lambda a: np.nanmean(corr_cols(predict(X_va, a), Y_va)))
    pred = predict(X_te, best)
    r_channels = corr_cols(pred, Y_te)
    te_frames = utt_of_frame[split_of_frame == 2]
    r_sentences = np.array([np.nanmean(corr_cols(pred[te_frames == k], Y_te[te_frames == k])) for k in test_utts])
    return best, r_channels, r_sentences

rows, per_sentence = [], {}
def record(feature, layer, X):
    alpha, r_ch, r_sent = probe(X)
    rows.append(dict(feature=feature, layer=layer, alpha=alpha, mean_r=float(np.nanmean(r_ch)),
                     **{c: float(v) for c, v in zip(CH_LABELS, r_ch)}))
    per_sentence[(feature, layer)] = r_sent

record("mfcc", -1, X_mfcc)
for kind in models:
    for l in tqdm(range(N_LAYERS), desc=f"probing {kind}"):
        record(kind, l, np.load(layer_path(kind, l), mmap_mode="r"))

results = pd.DataFrame(rows)
results.to_csv(os.path.join(RESULTS_DIR, f"{SPEAKER}_probe_results.csv"), index=False)
display(results[["feature", "layer", "alpha", "mean_r"]].round(3))

## Tests of the predictions

In [ ]:
pre = results[results.feature == "pretrained"].set_index("layer")
rnd = results[results.feature == "random"].set_index("layer")
mfcc_r = float(results.loc[results.feature == "mfcc", "mean_r"].iloc[0])
best_layer = int(pre.mean_r.idxmax())

def paired(a, b):
    d = np.asarray(a) - np.asarray(b)
    d = d[np.isfinite(d)]
    return (np.nan, len(d)) if len(d) < 6 else (stats.wilcoxon(d).pvalue, len(d))

p1, n1 = paired(per_sentence[("pretrained", best_layer)], per_sentence[("mfcc", -1)])
p2, n2 = paired(per_sentence[("pretrained", best_layer)], per_sentence[("random", best_layer)])
layers_beating_random = int((pre.mean_r > rnd.mean_r).sum())
peak_position = best_layer / (N_LAYERS - 1)

print(f"(P1) best pretrained layer {best_layer}: r = {pre.mean_r[best_layer]:.3f} vs MFCC r = {mfcc_r:.3f}; "
      f"paired Wilcoxon over {n1} test sentences p = {p1:.3g}")
print(f"(P2) pretrained > random at {layers_beating_random}/{N_LAYERS} layers; at layer {best_layer}: "
      f"{pre.mean_r[best_layer]:.3f} vs {rnd.mean_r[best_layer]:.3f}, p = {p2:.3g}")
print(f"(P3) peak at layer {best_layer} of {N_LAYERS - 1} ({peak_position:.0%} of depth); "
      f"final layer r = {pre.mean_r[N_LAYERS - 1]:.3f}")

## Figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1.1, 1]})

ax = axes[0]
ax.plot(pre.index, pre.mean_r, "-o", color=FEATURE_COLOR, lw=2, ms=6, label="HuBERT (pretrained)")
ax.plot(rnd.index, rnd.mean_r, "-o", color=RANDOM_COLOR, lw=2, ms=6, label="HuBERT (random init)")
ax.axhline(mfcc_r, color=MFCC_COLOR, ls="--", lw=2, label="MFCC baseline")
ax.scatter([best_layer], [pre.mean_r[best_layer]], s=140, facecolor="none", edgecolor=FEATURE_COLOR, lw=2, zorder=3)
ax.set_xlabel("layer (0 = input to the first transformer layer)")
ax.set_ylabel("mean correlation with measured EMA (test)")
ax.set_title("Articulatory information by layer")
ax.set_xticks(range(N_LAYERS))
ax.legend(frameon=False, loc="best")

ax = axes[1]
arts = ARTICULATORS
x = np.arange(len(arts))
def per_articulator(row):
    return [np.nanmean([row[f"{a}_x"], row[f"{a}_y"]]) for a in arts]
best_row = results[(results.feature == "pretrained") & (results.layer == best_layer)].iloc[0]
mfcc_row = results[results.feature == "mfcc"].iloc[0]
b1 = ax.bar(x - 0.2, per_articulator(best_row), width=0.38, color=FEATURE_COLOR, label=f"HuBERT layer {best_layer}")
b2 = ax.bar(x + 0.2, per_articulator(mfcc_row), width=0.38, color=MFCC_COLOR, label="MFCC")
ax.set_xticks(x, [ARTICULATOR_NAMES.get(a, a) for a in arts], rotation=30, ha="right")
ax.set_ylabel("correlation (mean of x and y)")
ax.set_title("Per articulator, best layer vs MFCC")
ax.legend(frameon=False, loc="best")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#e6e5e0", lw=0.8)
    ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, f"{SPEAKER}_probing.png"), dpi=150, bbox_inches="tight")
plt.show()

## Summary (paste this back)

In [ ]:
summary = f"""HuBERT articulatory probing - {MODEL_NAME}, MOCHA-TIMIT speaker {SPEAKER}
{len(plan)} sentences ({TOTAL / 50 / 60:.1f} min), test sentences: {len(test_utts)}, {len(CH_LABELS)} EMA channels.
(P1) Best pretrained layer {best_layer}: r = {pre.mean_r[best_layer]:.3f}; MFCC: r = {mfcc_r:.3f}; Wilcoxon p = {p1:.3g}.
(P2) Pretrained beats random init at {layers_beating_random}/{N_LAYERS} layers; layer {best_layer}: {pre.mean_r[best_layer]:.3f} vs {rnd.mean_r[best_layer]:.3f}, p = {p2:.3g}.
(P3) Peak at layer {best_layer} of {N_LAYERS - 1}; final layer r = {pre.mean_r[N_LAYERS - 1]:.3f}.
Per articulator at layer {best_layer}: """ + ", ".join(
    f"{ARTICULATOR_NAMES.get(a, a)} {v:.2f}" for a, v in zip(ARTICULATORS, per_articulator(best_row)))
print(summary)
open(os.path.join(RESULTS_DIR, f"{SPEAKER}_summary.txt"), "w").write(summary)

## Reading the results honestly

- **A linear probe shows that information is *decodable*, not that the model *uses* it.** Testing use needs interventions, such as ablating directions and checking the model's behaviour. That's the planned next step with sparse autoencoder features.
- **MFCC is the bar to clear.** If HuBERT beats MFCC by only a small margin, most of what the probe finds is acoustics, not learned articulatory structure.
- **The random-init model matters.** Even untrained convolutional features carry acoustic information, so any layer where pretrained ≈ random has learned little beyond that.
- **Expect the random-init line to be almost flat.** Untrained transformer layers barely change their input (in testing, layers differed by only about 1–2%), so every random layer scores about the same as its input layer. That's expected, not a bug.

## Limitations

- One speaker (rerun with `SPEAKER = "msak0"` to check that the layer profile generalises).
- Silent stretches at the start and end of sentences are included, where the articulators barely move.
- Linear probes only; per-sentence mean removal discards absolute articulator position.
- No correction for head movement beyond what the dataset provides.

## Next step

Train a sparse autoencoder on the best layer and test whether individual features track specific articulators, using ablation with a random-direction control, as in the Gemma experiments.